# Rotary position embedding (RoPE)

Implement the rotary position embedding that is applied to queries and keys before attention.

```python
def apply_rope(x, positions=None, base=10000.0):
    # Your code
    return out
```

- `x`: float tensor of shape `(..., T, d)` with `d` even (queries or keys; any number of leading axes).
- `positions`: `None`, meaning positions $0, 1, \dots, T-1$, or a 1D `int64` tensor of length `T` giving the position of each row (for example `torch.arange(100, 100 + T)` when new tokens are appended after 100 cached ones).
- `base`: the constant in the frequencies.

Coordinates $(x_{2i}, x_{2i+1})$ form pair $i$, for $i = 0, \dots, d/2 - 1$. The row at position $m$ has every pair rotated counterclockwise by the angle $m\,\theta_i$ with $\theta_i = \text{base}^{-2i/d}$:

$$\text{out}_{2i} = x_{2i}\cos(m\theta_i) - x_{2i+1}\sin(m\theta_i), \qquad \text{out}_{2i+1} = x_{2i}\sin(m\theta_i) + x_{2i+1}\cos(m\theta_i).$$

Return `out` with the same shape and dtype as `x`. Gradients must flow to `x`. Do not modify `x`.

**Example.**

```python
x = torch.ones(2, 4)                 # T = 2 positions, d = 4
apply_rope(x)
# tensor([[ 1.0000,  1.0000,  1.0000,  1.0000],      position 0: unchanged
#         [-0.3012,  1.3818,  0.9900,  1.0099]])     pair 0 turned by 1 rad, pair 1 by 0.01 rad
```

A useful self-test: with `q = apply_rope(a)` and `k = apply_rope(b)` for random `a`, `b`, the score `q[m] @ k[n]` depends only on `n - m` when `a` and `b` have the same vector at every position.

**Restrictions.** Only `import torch` and `import math`.

**Constraints.** Up to 64 leading slices, $T \le 4096$, $d \le 256$. Time limit 1 second per call on CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem4_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem4_submission.py
import math
import torch


def apply_rope(x, positions=None, base=10000.0):
    # Your code
    return out


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem4_submission as submission
importlib.reload(submission)
# e.g. submission.apply_rope(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9W41z27ix/1fwdDMN6cqMrDjp1Tnfq3Pnu6aTr4ndaTuWhweJkMyaInkEGct1/b+/3y4AEtSHE+fal8zYJAgsdhf7jfXdIE0GR2LQHDzfL0eHg6EYLFUtMXS3+aVO60zR2MeiltWtKAud1mmRC7WcqiRJ84UIPhYfTkOanCmti9xByEbPGXaaN7XSGD14jtcknc/TWZPVtxh5hgGV1xU9D2RZZrdxVZSKlxVJk6k4l0vevqyKaaaWh7FupstUa6BAs2SWFTcqidNlWVQ1bXIxqItqdsUgZH01uMTDVOa5Isou7FucFUVJs+cy04rIVLqO63SpiqaO6cN4RKNyYUA6qiOHHe8R6SydgQPdwLQqZDKTuqZR2ksXTTVjAooqXaS5zGhyKat6jMG6atQ93udpxgy6G1zJKgcTo5I5MuH/f1ZZqSot6itZC8sHsahkQoOGcBH8Mq+KpbDL3ejeL2E0ySf5iZ0uUi1mRV6D5U+bPK310+/o1/dPLVS8l2ny/VMzG0hE4nUtEjVPAVUQj8S8yWfECS3oZBIejPcMbrW8VnhSk7w7JGHOUfyCoV8icfr3k7cf3pyeiSzVNZGkDFgCpoW+Km5ygUUfm1wEn1KdAqvwpTgjcLWoGmyrPimIIS0CaT/JNGsqrKxkqpWgV3GT1ldCCsDTcqHETZXWIFjMi0q8UvmR0PJW3BC6N1Lja5EvhiInqCJplqW4SpME09O8bGpN3DOnkFuWzorytn0hgSH+2tcc628FoOYljX4jfigyORU/fPirFrJSQpOsVsSrnFB5osXbsfhQFUOhC5EVM5kRRIiOFssGTJkqYipYVJDKpPicpcSHKei/wlHO5QxiF9FO42hEhyvFoqHzD8zOGrAytV9fVUomQpcKB4ZZVdEsrrJbSEs2F8AFWMyKihhdKWJ6LdI5IYhX1ipLh1UPYsrZh9PTH+OfTn44f/9RHNPuRPAkn2USu9MxBKermSpJUsKjSS7wr8QnMwsCBY6rJIAsJkOx1As3B/vmBTE5T+wI/esON6C5HZA4L4OVWwsr4q1xJ0Rq6Y3OQX+a61rmMxWshuZ7dK5yXVSht5y3VXVT5WIVJbCOs6sgjGZlg598zEFoJismU7zm7U6rqqg8KIZkD1ZeRlLLqpK3wLujQ1/JUgWLoh5CKHP6Cfk8ngwg2TCUk4HDTIPXdVNmKiDKMT+MeKnFhXmKOW4SwQqHYj4Z3BHAexy4NnuJOw27o1almtWQiTtv/j226zCbZYV+CLOhqOoiOz5Q+y+GQtrHbx2+0KwboNwiC8N4C0LnWSEJMRrnPfsfPGIWhj4i6cY87iDHEGCnh9toc5+YvHYDHEiqYd7SGquBR5YFoYClICn0v93Yb/72ZEllypYwpzVpPm+BW0Gm884yy0Qww7KLflh20Q9f8FRVgWPMCEJuKVf0S051sBD74iYMQ4INQtN/KaHgvMSIlG+LqvT5xIZOfJIZ7IMIMlktyOwqElhxh19H0bPFfdg7e/VrI7MvkMq+UpmphHSesKQPQ+YokJB1XdnvkwEfRwdki7h4AtLN+a+LhS8ajH9s+QDkNpTJ52oLgU7laB1jCANhy8R3MGCM7/Dpf6oeajQLQ73DSHXMKsLHQU9653nwht6JdEs28KdRsMICDqOY46047qGzgPmjk7O7PnGTnjiofUxjncsSnrz+/zXNWZGrRxplbOmkckUiyb59EMIlJqy9fXyCJJ2BkRS4hD4+feJ62BHAIFGqPD5HpOcJmEXwHOxbR28TQNhzITQUEUz+tuoxHscSyKGYfj3f5Rfx3UwxesH7fS3bJbGd4ei+KbAbSbOJDqbhNluD5Z2d2bq8tRNT+8hn29dsomBorB0kO/fPysFhCF7wUsTLBkqB6CaY50OxB3MKzdjbu76hJ4eJCR1/gAcQ85x3nlOASjQgxkEUuIB6yRwhY7VolgjK4UJEmcmZilzUyV6f9Al28aLTLBlyPCtpAe14aWYiOiPnkQfrCJnPtCSFIINgHdJShVBGVRL+7V9pGZg1vEdPwNmetMJljcgtEhvIYqIowE/nKUWWFDJ3tHxzl4rfi4P7l+KmqK4pqpcsvIKOD/Foay8tk4F8x2EK4tM80IpCQWD1pVx+6bMY1tUPubdE0pw+wh9qxN6ggBjEse6TdiWCd5ND0VbvAaKiFAwJ0lITSRyHv5UzMcMGtAREmqibshUO7Ck4QHaxP20SWFIa5hgcYXlKCVRTQp1EfZPOFAEEh1J46GJuNRg4O45qFiEamSNeIvdNELCiwbdInNOHtMIwiMme0BbkF8xmxGGV9MSqHlE0CQgRsst5zGBU5azNZ0QpqbcvRoxS23DE0nss7CGKp8JPGzqRjEkUK9KGYOzLHQ4R23x3bCGtmaIpcprrbkiOh+J6jN369pFRHq4N9ij5PCsYT2LEmNkw9oaZDcjYgoRc7A52+PGmTxDrUV0ULDdI/pP6KBrP7xHlQ8rUS5t1tULMwgS1Y0CiySmfv7Osses4h0Ri5454yz/eEpkl0uA5RGtBAaxJMu/sOTEg0o4HFbTKF9BOlbgD6xIcnGRSLMHtuYT4xe3ECRdsGi1lWsRcYehVOX6mEWgAMOuKBykbjas0S0jrZlQZQcKcki7QzL+cvX8HXHPWHF0nwJB1tbytrzCy/1q4fcR3tsQRJ2n1vfiu2wMf8Q6t/DeP1d9vpPxQNi/9x8Gu6iydtkPmF0aipk6zbrhoH/+pi65IUMr6yl+u00Uuu2X6Vq+VF9xLBb8wlbPrdgRSnFOxwNQa9CwFmblqzR0GrtN6P1OYJg6iF9GBjen0EdWJllSsG7JJQXZhnY8zfqVMweXFJHd7RPM0g0S712AyANpFpSgDtJUWRKLR3l/z65xqOLrIqKJScAUAG6YlnVq0ZyPFP59+PIXmWF5EH/A7iGMqg8VxGMFw0XJEdaWsgNkkB1cimhxBWlVVByMoA8IHAhMC4LbaF9eCxDfw1b/KI3F6OBpzJezN307+cRa/Onn37vRHYHA3GcS2eBjHRAwIc+dpXtVKzewT4nzzNCuWJXCdDO4n+Zv37z/E797/eHoGcAFkJfqpgK7Sw98guMo8nujbfNZ+eIMY8gfAMG9nynv5EUFm9/azysk/F9Xpqgz9Esu5qcV8rsoSS2R6y4BkrFnC3sBtqVZjOVV0gPxqBEU2s3h2pWbXOtDVjI64ln1/+9HoO0XDcFbEx+LaMGepF2DMS5hFigBNjY/VRtdP2tPhYMiWEslR1VVqyoo9F2Wc0MXltmi2rhQCO+YSpAT5NRDtx6Fnt8jPVxyHUlFObcaHFxbtI0gL14Et9kdkJjUvtylykRs7c6ci+p0X90f0jNlIfu4tgrYYTS4B/IpgmKEnaxVq4tHFpUV0Kttx9pO1czlcDo0bzcC88Sn0ITa1UjtOIo6QEbqbJlAVisQqEygYwIBg5geU0VDQnU6pHA9yrjl2U1lEgm034Lq4x/OlrJdN5o/MuToCU022Gcy/kdl1QIex5rq9MJ2mG3k2qcF6VtGGsjQxYnSPNh2YLakgXg4kT6KgfEmlBfMaYTcwkiLHYInIE+aIkjkCvmRkzUGER9t9o3cWkUwSu0e4Obk7G29epMsspeOmPS/2Dy63LOwOL2rKhKLuLWv9JDF7mI0/weqtU4MDxVkxH23hHeR3CrWFl7zCMhLPv4GL6xwEtDUueKzrcWCbBKwt3cW8r1rKdD6W6SdOd9ZpXxMIxoMU7YugvvMs8kMA02Q3Vdsnbduxc1bre7LiR7IsVZ4EruJjFk0GxuQZ8/6/fX5t24W94Ks0f2+92EmzONHkg0JTW1lbEBV23ltZv0XIuCHTxgY9Ejkg5gmkB5KCRAtqp/l3UYDREiPyNMoKQfOifxaUnrKBDbx9wnC7dfk9eRQwxoG6ezIUTwwQp1GkgE+wH/LexRMqxd5vryduw59KFlvQ1w73Dtg37Q3iVOaaeDu7sh6CQ8HOQXD5WUkksX/n3+FLk41mN/JW75srVWQyiAvTFhAJpYVW5Nnt0O2JzKVzNYGRAlNEsqEUBR23IlMLpOY3FDYJnhlFUeg5MwRlIinoUAzgq5TzS3MKgec0f8fO0fPCBl3rfCGG/xaB50t/J3pxoSdC2OHRogMMtLD8IfiBDrfIDiCHv+WE88Lt0V7JdqcMzPvEx0aJ4qI08WS/2kbTjZKtqd6XEEwZJeTiT8LBppTV0xsRmMjJ7nAxuvSFewv5j2FBt+tDxLNx2yCZR7+OYBMmsbD3CMUwqTEJdaWuVM5JbBs78qr/LP2WsvsHMvV5lQJMdhtzJBsABBLdmLKpfkT/Plf7jGei9KxKOasQJUSVmhmEvYnmCam7vqaEDeqqaJ7my/p+hWlKZRWXt0bIm/Fcx/U0UFEctx+Q8tlgk4Aj0pyzU5/TPgCBk5pT+qlczEeZX0uCjUpvqGRCBSuyslybtEKHnxSOeVF7b4iOggWGZvN1Woe+ZSbdmPANifLuRyj6v7/jXe9pvmP2EpF40OZisLTrWS5lsrJafLpoQ0RHys6Z40svJbacogD8WLRTnrlwHXKPcSo4RFkhEx0EjMRTEhvSCS5FDAgefBZVMgJn7JA+EUSLjD/BcYOricdiUy9Mwjix/TUTbqbR3aspzBiRtZk0SSINvCtwBD78iw7aJaGzIxVtFZ0iWFJBfWGwumRvQW+sJZd+OOtSOhfPaq47bmzrWwmuWgTMTmoO0YGZ7nt5IyWu5DkH0mkRndVUPnn93jHPFHki8yuwb2evfz558/Ht0GbpdirMWczFXXcL7NmyfpcS8XI8crj0s2NugumKVTjOJK0UlM8cRgBM18Msh6OqU9qicmi+Pn/99vRj/PH05M2wxW4tzNGlIunpl8IiGo2pNmMqO1Sd5CsTjyKvy8uIRleYwzF5pmptv2a6uZ2FxRvS1gH9WF8IjTHz9AV95oTh0gj+JkmsQ6qKqAYUm2VkecKvYNyoX5uwZZdeScIKolWOS2PNDI1kgdmqscWti+Ka7xTEnTuPo2hEFWBnuuwur6RWbYXI1EC6ktirN6ej0UGHwKPI2IWxvRhiNOkmhjrECntQNgh6wB+1ar0BehujnF2hfSHPJFF8Ex8gAT8cjUZHl1+tyNyT2K/j8JDf32HmuLtid6MJWobmU/j5w+14lRTKBE2m2Q7ehUDc+3n7I0lYPE4nJwPT9Eda2LqMtg+wpdv2EX5e9xae8i126ZKBZmfZdj9b2HJZnplClsG1Drog3po98jUUNARIVObGsNMT2fZPstJujyit1VIH3D2T970CW1V3708XKBJoBPPQL3jBStAlJIcktONa8MzumKS/amC3jFBQtMJdBLkjbc3gknlO80b55zczDtbYwyMGcvH86BL2m++Fga2tUgs2j5OBBe0me7vRV6oGU9qzLU2x9tZc+tDQKBrdP+ZObLPx4Su8zm/1PPZybtMqf5VJs4dw0XGONJUC783ejQ0L3i02sYfRcTbWWZEv3L3Mus0WQZqbLjOO5oeUQMxlJdzNYOgbAbs5X26s1bR/I8EtzhReb+lVeZQv+U/h4vcZ8H1F8oU+pIPVCjjBqyDFSbDronYonm1zbzacvXQJWdUatke7onaBiVWPKXUMutj1oaiUjcq0KLJgAzHzjUDVPcklcHUPXLvCJSg7XQq3Nc+FS3iMdYtjym/oouzIXQ9QumNudltnQXe6tnxGHfjdDaZpPNp1WTfki+2haTkdCtvW57XaDG1XCKHW9pHD8lsLrlZyWZIlvPRuqdUcGZpt4HJN/PqY0o6hmEKijw9wOjB8ziytAHAV4TDJBVgunQ9FwsOM0sX+GFlN+3JgTxbAMYeuwwLTynBuOjXbTakqQvuaLLMdjuqCF/XaLlb9djY+x6Gpv3ctOwDhG1Nu7uk6KRLx9KkYr1tbSg6XYo9JF3t7Itgf4y2Fs18vLyPTIoLoAKNZoQMJb8MvmkqW4Ua14iKKoiFhyQBJtFfrQ3tiBjVbG6YWIfqkPwPRzNsKVePbDqiz3WURzilNuAbZgM9X2Q4RoUfHyGtqUL27b4PA3vGSt6fJHtOvb0gj3SSj891rV6jGcTwAgL5b40JPnqT4nWigJer+hobJoq4dl4JyjzlWtcJrmkGZ8vsjglc2db87mqpQCfeIHrNa0OO2VdxLaibetWt6PaR2cddma3qiGZ3tSjoUVjV9BnUKRCPhdgq6nvTDrif9+QMVMt98cCRh+e+JiClWY3cdjIfiMDSdAbxCBOf0JxDGSBz2W6gZ8pRK4yqJTZuwvwFZGwOZmmZyAv1sKF4MxbdDsXBX/8dmRtsKEITRUubUrEhtNcGBi4T7Am0brIUHcwtqBTW0xcTLh9B6zuu/FKHxLoT4EHGodELO9kajdZRaAYhd39vnOXb45dg924Vdp8ltB2PQjQFVEsgD+jEOOUlqTYXZUBq7ayc+C9fpqlQGNf2k4pK0s6pvfbIWLVm78D60aFOz6hoPDnA4rF0WEy4bvejxZBEOH7/EbPirKY54liWQSOJKwAGxoPVF23xtV1xvrpg+vILc+K/iT+I6OvesD/3Rw9kFWH54Cb9xdgEx/BaxzncC6vxH17iEGQA5MjP+MBR/aGfQoTqmC8f0I6HpT5001bO545c6QhGiwvYCwlLQXydSErx+epZBMXbl3rX0M3JphPLzXH740J/vElYLybTctjZ1pf3YAdnXrw3SL80Ix4HX1mx8x9oZrfS66e8BGNp6AFF+ZHppje33ShcUHtN1xwrf+Y/MhGxqapbtmn0JblncwEpEulkCS7oBuJFVEvQcw0pHZlNy5yt/b+o2opUAtLc3DkVQFcb/aXGtVCkylS/qK83dkqYHeLwKd/6h0tox85/IPXiuZP1HfwSMg/EjjPSLsMd52918EI2GYsNrP+CvJwP+wx0xg/Xc8NitR72A8JFSHFB5+PBbRvj55WXP0e6cY+ioufs/2JwRhruw6Bj8rGMwWa3B/f3/AdY0bqY=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
